# Cross-Platform Community Evolution: Step 2 Data Cleaning & Preparation

**Project**: Show Reel Media Group Community Evolution  
**Dataset**: 987,115 raw comments (Instagram: 573,377 | Facebook: 394,084 | TikTok: 19,654)  
**Objective**: Unified Python processing pipeline with three optimized export formats

## Output Formats
- **comments_llm.jsonl**: Raw text + metadata for LLM token efficiency
- **comments_ml.parquet**: Flattened columnar features for XGBoost/LightGBM
- **comments_gml.parquet**: Edge-list format for PyTorch Geometric GNNs

## Section 1: Imports and Configuration

In [ ]:
import json
import logging
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path
import re
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import emoji
from scipy.stats import entropy as scipy_entropy

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

## Section 2: Type Definitions and Emoji Taxonomy

In [ ]:
class PlatformType(Enum):
    INSTAGRAM = "instagram"
    FACEBOOK = "facebook"
    TIKTOK = "tiktok"

class EmojiCategory(Enum):
    LOVE = "love"
    CELEBRATION = "celebration"
    HUMOR = "humor"
    INQUIRY = "inquiry"
    OTHER = "other"

@dataclass
class RawComment:
    """Unified raw comment structure"""
    platform: str
    author_id: str
    media_id: str
    comment_text: str
    reply_to_comment_id: Optional[str]
    timestamp: str
    original_from_id: Optional[str] = None
    original_uid: Optional[str] = None

@dataclass
class ProcessedCommentML:
    """ML-optimized columnar representation"""
    comment_id: str
    author_id: str
    media_id: str
    platform: str
    text_length: int
    word_count: int
    emoji_count: int
    unique_emoji_count: float
    emoji_entropy: float
    emoji_variety_ratio: float
    emoji_per_word_ratio: float
    url_count: int
    mention_count: int
    hashtag_count: int
    exclamation_count: int
    question_count: int
    avg_word_length: float
    has_numbers: int
    has_links: int
    timestamp: str

@dataclass
class ProcessedCommentGML:
    """Graph ML edge-list representation"""
    comment_id: str
    author_id: str
    media_id: str
    reply_to_comment_id: Optional[str]
    platform: str
    timestamp: str

# Emoji Taxonomy Mapping
EMOJI_TAXONOMY = {
    EmojiCategory.LOVE: ['❤️', '💕', '💖', '😍', '🥰', '💗', '💝', '💞', '👍', '💯'],
    EmojiCategory.CELEBRATION: ['🎉', '🎊', '🥳', '🎈', '✨', '⭐', '🌟', '🔥', '👏', '🙌'],
    EmojiCategory.HUMOR: ['😂', '🤣', '😄', '😁', '😆', '🤪', '😜', '🤩', '😏', '💀'],
    EmojiCategory.INQUIRY: ['❓', '❔', '🤔', '🧐', '😕', '🤨', '💭', '🙄', '😐', '😒']
}

# Reverse mapping for fast lookup
EMOJI_TO_CATEGORY = {}
for category, emojis in EMOJI_TAXONOMY.items():
    for e in emojis:
        EMOJI_TO_CATEGORY[e] = category.value

## Section 3: Utility Functions (Emoji Extraction & Feature Engineering)

In [ ]:
def compute_shannon_entropy(emoji_list: List[str]) -> float:
    """Compute Shannon Entropy of emoji distribution.
    
    High entropy = diverse emoji usage; Low entropy = repetitive usage.
    Returns 0 if no emojis present.
    """
    if not emoji_list:
        return 0.0
    
    counts = Counter(emoji_list)
    frequencies = np.array(list(counts.values())) / len(emoji_list)
    return float(scipy_entropy(frequencies))

def extract_emojis_safe(text: str) -> List[str]:
    """Extract emojis using emoji library to preserve ZWJ sequences.
    
    Properly handles composite sequences like skin tone modifiers,
    gender variations, and coupled glyphs.
    """
    try:
        return [e['emoji'] for e in emoji.emoji_list(text)]
    except Exception as e:
        logger.warning(f"Emoji extraction failed: {e}")
        return []

def categorize_emojis(emoji_list: List[str]) -> Dict[str, int]:
    """Categorize extracted emojis into semantic buckets."""
    categories = defaultdict(int)
    for em in emoji_list:
        category = EMOJI_TO_CATEGORY.get(em, EmojiCategory.OTHER.value)
        categories[category] += 1
    return dict(categories)

def compute_emoji_features(emoji_list: List[str], word_count: int) -> Tuple[float, float, float]:
    """Compute emoji features: variety_ratio, entropy, per_word_ratio."""
    if not emoji_list:
        return 0.0, 0.0, 0.0
    
    unique_count = len(set(emoji_list))
    variety_ratio = unique_count / len(emoji_list) if emoji_list else 0.0
    entropy_val = compute_shannon_entropy(emoji_list)
    per_word_ratio = len(emoji_list) / word_count if word_count > 0 else 0.0
    
    return variety_ratio, entropy_val, per_word_ratio

## Section 4: Regex Pre-compilation & Advanced Text Preprocessing

In [ ]:
# Pre-compile all regex patterns for linear-time O(n) batch processing
REGEX_PATTERNS = {
    'url': re.compile(r'https?://\S+|www\.\S+'),
    'mention': re.compile(r'@\w+'),
    'hashtag': re.compile(r'#\w+'),
    'exclamation': re.compile(r'!'),
    'question': re.compile(r'\?'),
    'numbers': re.compile(r'\d'),
    'whitespace': re.compile(r'\s+')
}

def advanced_text_preprocessing(text: str) -> Dict[str, Any]:
    """Advanced preprocessing with linguistic feature extraction.
    
    Returns dict with counts and derived metrics for ML features.
    """
    if not isinstance(text, str) or not text.strip():
        return {
            'text_length': 0,
            'word_count': 0,
            'emoji_count': 0,
            'unique_emoji_count': 0,
            'emoji_entropy': 0.0,
            'emoji_variety_ratio': 0.0,
            'emoji_per_word_ratio': 0.0,
            'url_count': 0,
            'mention_count': 0,
            'hashtag_count': 0,
            'exclamation_count': 0,
            'question_count': 0,
            'avg_word_length': 0.0,
            'has_numbers': 0,
            'has_links': 0
        }
    
    text = text.strip()
    text_length = len(text)
    
    # Extract emojis using safe method
    emojis = extract_emojis_safe(text)
    emoji_count = len(emojis)
    unique_emoji_count = len(set(emojis))
    
    # Remove emojis for word-level analysis
    text_without_emoji = ''.join(c for c in text if c not in emojis)
    
    # Word tokenization
    words = REGEX_PATTERNS['whitespace'].split(text_without_emoji.strip())
    words = [w for w in words if w]  # Remove empty strings
    word_count = len(words)
    
    # Compute emoji features
    variety_ratio, entropy_val, per_word_ratio = compute_emoji_features(emojis, word_count)
    
    # Pattern matching with pre-compiled regex
    url_count = len(REGEX_PATTERNS['url'].findall(text))
    mention_count = len(REGEX_PATTERNS['mention'].findall(text))
    hashtag_count = len(REGEX_PATTERNS['hashtag'].findall(text))
    exclamation_count = len(REGEX_PATTERNS['exclamation'].findall(text))
    question_count = len(REGEX_PATTERNS['question'].findall(text))
    has_numbers = 1 if REGEX_PATTERNS['numbers'].search(text) else 0
    has_links = 1 if url_count > 0 else 0
    
    # Average word length
    avg_word_length = np.mean([len(w) for w in words]) if words else 0.0
    
    return {
        'text_length': text_length,
        'word_count': word_count,
        'emoji_count': emoji_count,
        'unique_emoji_count': unique_emoji_count,
        'emoji_entropy': entropy_val,
        'emoji_variety_ratio': variety_ratio,
        'emoji_per_word_ratio': per_word_ratio,
        'url_count': url_count,
        'mention_count': mention_count,
        'hashtag_count': hashtag_count,
        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'avg_word_length': float(avg_word_length),
        'has_numbers': has_numbers,
        'has_links': has_links
    }

## Section 5: Unified Pipeline - Platform Normalization & Processing

In [ ]:
class UnifiedPipeline:
    """Production-grade data preparation pipeline."""
    
    def __init__(self, output_dir: Path = Path("./output")):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.processed_count = 0
        self.error_count = 0
        self.errors = []
    
    def normalize_raw_comment(self, record: Dict[str, Any], platform: str) -> Optional[RawComment]:
        """Normalize disparate platform schemas to unified RawComment."""
        try:
            platform_lower = platform.lower()
            
            if platform_lower in ['instagram', 'facebook']:
                author_id = str(record.get('from_id', ''))
                reply_to_comment_id = record.get('parent_id')
                media_id = str(record.get('media_id', ''))
            elif platform_lower == 'tiktok':
                author_id = str(record.get('uid', ''))
                reply_to_comment_id = record.get('reply_id')
                media_id = str(record.get('video_id', ''))
            else:
                raise ValueError(f"Unknown platform: {platform}")
            
            if not author_id or not media_id:
                return None
            
            comment_text = str(record.get('text', ''))
            timestamp = str(record.get('timestamp', ''))
            
            return RawComment(
                platform=platform_lower,
                author_id=author_id,
                media_id=media_id,
                comment_text=comment_text,
                reply_to_comment_id=reply_to_comment_id,
                timestamp=timestamp,
                original_from_id=record.get('from_id'),
                original_uid=record.get('uid')
            )
        except Exception as e:
            logger.error(f"Normalization failed for record {record}: {e}")
            self.errors.append({"record": record, "error": str(e)})
            return None
    
    def process_comment(self, raw: RawComment, comment_id: str) -> Tuple[Optional[ProcessedCommentML], Optional[ProcessedCommentGML]]:
        """Process single comment into ML and GML formats."""
        try:
            # Text preprocessing
            features = advanced_text_preprocessing(raw.comment_text)
            
            # ML format
            ml_comment = ProcessedCommentML(
                comment_id=comment_id,
                author_id=raw.author_id,
                media_id=raw.media_id,
                platform=raw.platform,
                text_length=features['text_length'],
                word_count=features['word_count'],
                emoji_count=features['emoji_count'],
                unique_emoji_count=float(features['unique_emoji_count']),
                emoji_entropy=features['emoji_entropy'],
                emoji_variety_ratio=features['emoji_variety_ratio'],
                emoji_per_word_ratio=features['emoji_per_word_ratio'],
                url_count=features['url_count'],
                mention_count=features['mention_count'],
                hashtag_count=features['hashtag_count'],
                exclamation_count=features['exclamation_count'],
                question_count=features['question_count'],
                avg_word_length=features['avg_word_length'],
                has_numbers=features['has_numbers'],
                has_links=features['has_links'],
                timestamp=raw.timestamp
            )
            
            # GML format
            gml_comment = ProcessedCommentGML(
                comment_id=comment_id,
                author_id=raw.author_id,
                media_id=raw.media_id,
                reply_to_comment_id=raw.reply_to_comment_id,
                platform=raw.platform,
                timestamp=raw.timestamp
            )
            
            return ml_comment, gml_comment
        except Exception as e:
            logger.error(f"Processing failed for comment {comment_id}: {e}")
            self.errors.append({"comment_id": comment_id, "error": str(e)})
            return None, None
    
    def export_llm_jsonl(self, records: List[Tuple[str, RawComment]], platform: str):
        """Export to JSONL for LLM token efficiency."""
        output_file = self.output_dir / f"comments_llm_{platform}.jsonl"
        with open(output_file, 'w', encoding='utf-8') as f:
            for comment_id, raw in records:
                record = {
                    "comment_id": comment_id,
                    "text": raw.comment_text,
                    "author_id": raw.author_id,
                    "platform": raw.platform,
                    "timestamp": raw.timestamp
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')
        logger.info(f"Exported {len(records)} records to {output_file}")
    
    def export_ml_parquet(self, records: List[ProcessedCommentML], platform: str):
        """Export to Parquet for ML frameworks."""
        if not records:
            logger.warning("No ML records to export")
            return
        
        data = [asdict(r) for r in records]
        df = pd.DataFrame(data)
        
        output_file = self.output_dir / f"comments_ml_{platform}.parquet"
        table = pa.Table.from_pandas(df)
        pq.write_table(table, output_file, compression='snappy')
        logger.info(f"Exported {len(records)} records to {output_file}")
    
    def export_gml_parquet(self, records: List[ProcessedCommentGML], platform: str):
        """Export to Parquet for Graph ML frameworks."""
        if not records:
            logger.warning("No GML records to export")
            return
        
        data = [asdict(r) for r in records]
        df = pd.DataFrame(data)
        
        output_file = self.output_dir / f"comments_gml_{platform}.parquet"
        table = pa.Table.from_pandas(df)
        pq.write_table(table, output_file, compression='snappy')
        logger.info(f"Exported {len(records)} records to {output_file}")
    
    def process_batch(self, raw_records: List[Dict[str, Any]], platform: str) -> Tuple[List[ProcessedCommentML], List[ProcessedCommentGML], List[Tuple[str, RawComment]]]:
        """Process batch of raw records from a platform."""
        ml_results = []
        gml_results = []
        llm_results = []
        
        for idx, record in enumerate(raw_records):
            try:
                # Normalize
                raw = self.normalize_raw_comment(record, platform)
                if not raw:
                    self.error_count += 1
                    continue
                
                # Generate unique ID
                comment_id = f"{platform}_{raw.author_id}_{self.processed_count}_{idx}"
                
                # Process
                ml, gml = self.process_comment(raw, comment_id)
                if ml and gml:
                    ml_results.append(ml)
                    gml_results.append(gml)
                    llm_results.append((comment_id, raw))
                    self.processed_count += 1
                else:
                    self.error_count += 1
            except Exception as e:
                logger.error(f"Batch processing error at index {idx}: {e}")
                self.error_count += 1
        
        return ml_results, gml_results, llm_results

## Section 6: Testing with Mock Data (Instagram, Facebook, TikTok)

In [ ]:
def test_pipeline_with_mock_data():
    """Comprehensive test with mock Instagram, Facebook, and TikTok records."""
    
    logger.info("Initializing unified pipeline...")
    pipeline = UnifiedPipeline(output_dir=Path("./test_output"))
    
    # Mock Instagram records
    instagram_records = [
        {
            'from_id': 'insta_user_001',
            'media_id': 'insta_post_101',
            'text': 'Love this! ❤️😍 Amazing content! Check out www.example.com #awesome',
            'parent_id': None,
            'timestamp': '2026-03-01T10:30:00Z'
        },
        {
            'from_id': 'insta_user_002',
            'media_id': 'insta_post_101',
            'text': 'This is fire 🔥🔥🔥 Who else thinks this is great? @insta_user_001',
            'parent_id': 'comment_001',
            'timestamp': '2026-03-01T10:35:00Z'
        }
    ]
    
    # Mock Facebook records
    facebook_records = [
        {
            'from_id': 'fb_user_001',
            'media_id': 'fb_post_201',
            'text': 'Hahaha 😂😂 This made my day!!!!! Check this out: https://facebook.com/video #funny',
            'parent_id': None,
            'timestamp': '2026-03-02T14:20:00Z'
        },
        {
            'from_id': 'fb_user_002',
            'media_id': 'fb_post_201',
            'text': 'I totally agree! 👍 🎉 What do you think about this? #discussion',
            'parent_id': 'comment_002',
            'timestamp': '2026-03-02T14:25:00Z'
        }
    ]
    
    # Mock TikTok records
    tiktok_records = [
        {
            'uid': 'tk_user_001',
            'video_id': 'tk_video_301',
            'text': 'OMG this is so good! 🥳✨ Amazing work! Follow for more @tiktok_creator #viral',
            'reply_id': None,
            'timestamp': '2026-03-03T08:15:00Z'
        },
        {
            'uid': 'tk_user_002',
            'video_id': 'tk_video_301',
            'text': 'Best video ever seen! 🌟💯 Subscribe now!!!',
            'reply_id': 'comment_003',
            'timestamp': '2026-03-03T08:20:00Z'
        }
    ]
    
    # Process and export each platform separately
    platform_batches = [
        (PlatformType.INSTAGRAM.value, instagram_records),
        (PlatformType.FACEBOOK.value,  facebook_records),
        (PlatformType.TIKTOK.value,    tiktok_records),
    ]

    all_ml_records  = []
    all_gml_records = []

    for platform_name, raw_records in platform_batches:
        logger.info(f"Processing {platform_name} records...")
        ml, gml, llm = pipeline.process_batch(raw_records, platform_name)

        logger.info(f"  Exporting {len(llm)} LLM records → comments_llm_{platform_name}.jsonl")
        pipeline.export_llm_jsonl(llm, platform_name)

        logger.info(f"  Exporting {len(ml)} ML records  → comments_ml_{platform_name}.parquet")
        pipeline.export_ml_parquet(ml, platform_name)

        logger.info(f"  Exporting {len(gml)} GML records → comments_gml_{platform_name}.parquet")
        pipeline.export_gml_parquet(gml, platform_name)

        all_ml_records.extend(ml)
        all_gml_records.extend(gml)

    # Report statistics
    logger.info(f"\n=== PIPELINE EXECUTION SUMMARY ===")
    logger.info(f"Successfully processed: {pipeline.processed_count}")
    logger.info(f"Errors encountered: {pipeline.error_count}")
    logger.info(f"Success rate: {pipeline.processed_count / (pipeline.processed_count + pipeline.error_count) * 100:.2f}%")
    
    if pipeline.errors:
        logger.warning(f"First few errors: {pipeline.errors[:3]}")
    
    # Display sample output
    logger.info("\n=== SAMPLE ML OUTPUT ===")
    if all_ml_records:
        sample_ml = all_ml_records[0]
        logger.info(f"Comment ID: {sample_ml.comment_id}")
        logger.info(f"Platform: {sample_ml.platform}")
        logger.info(f"Emoji count: {sample_ml.emoji_count}, Entropy: {sample_ml.emoji_entropy:.3f}")
        logger.info(f"Word count: {sample_ml.word_count}, Text length: {sample_ml.text_length}")
    
    logger.info("\n=== SAMPLE GML OUTPUT ===")
    if all_gml_records:
        sample_gml = all_gml_records[0]
        logger.info(f"Comment ID: {sample_gml.comment_id}")
        logger.info(f"Author → Media edge: {sample_gml.author_id} → {sample_gml.media_id}")
        if sample_gml.reply_to_comment_id:
            logger.info(f"Comment reply edge: → {sample_gml.reply_to_comment_id}")

# Execute test
test_pipeline_with_mock_data()